<a href="https://colab.research.google.com/github/SeanNamUIUC/Semiconductor-AnomalyDetection/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================
# EDA for Semiconductor Sensor Dataset
# =====================================


# 1. import numpy, pandas, matplotlib, seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 2. matplotlib inline 설정
%matplotlib inline


In [53]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
!ls

drive  sample_data


In [ ]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Semiconductor-AnomalyDetection/data/raw/secom.data"
LABEL_PATH = "/content/drive/MyDrive/Semiconductor-AnomalyDetection/data/raw/secom_labels.data"

X = pd.read_csv(DATA_PATH, sep=" ", header=None)
y = pd.read_csv(LABEL_PATH, header=None)

print(X.shape, y.shape)

(1567, 590) (1567, 1)


In [ ]:
!find /content/drive/MyDrive/Semiconductor-AnomalyDetection -name "secom.data"

/content/drive/MyDrive/Semiconductor-AnomalyDetection/data/raw/secom.data


In [35]:
#Data loading


# 1. 데이터 파일 경로 설정
data_path = "/content/drive/MyDrive/Semiconductor-AnomalyDetection/data/raw/secom.data"
label_path = "/content/drive/MyDrive/Semiconductor-AnomalyDetection/data/raw/secom_labels.data"
#dataset description: https://archive.ics.uci.edu/dataset/179/secom

#label data format : [label]  [timestamp(string)]

# 2. pandas로 데이터 로드
df_data1 = pd.read_csv(data_path, header=None)
df_data2 = pd.read_csv(data_path, sep=" ", header=None)
df_data3 = pd.read_csv(data_path, sep="\s+", engine="python", header=None) # 공백하나이상일시 컬럼

df_label1 = pd.read_csv(label_path, header=None)
df_label2 = pd.read_csv(label_path, sep=" ", header=None)


# 3. 데이터 shape 출력
# print(df_data1.shape)
# print(df_data2.shape)
# print(df_data3.shape)
# print(df_label1.shape)


print(df_data2.shape)
print(df_label2.shape)


# print(df_label2.shape)
# print(df_label2.tail())
print(df_data2.head())
print(df_label2.head())
df_data2.info()
df_label2.info()



<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-661850304.py:14: SyntaxWarning: invalid escape sequence '\s'
  df_data3 = pd.read_csv(data_path, sep="\s+", engine="python", header=None) # 공백하나이상일시 컬럼


(1567, 590)
(1567, 2)
       0        1          2          3       4      5         6       7    \
0  3030.93  2564.00  2187.7333  1411.1265  1.3602  100.0   97.6133  0.1242   
1  3095.78  2465.14  2230.4222  1463.6606  0.8294  100.0  102.3433  0.1247   
2  2932.61  2559.94  2186.4111  1698.0172  1.5102  100.0   95.4878  0.1241   
3  2988.72  2479.90  2199.0333   909.7926  1.3204  100.0  104.2367  0.1217   
4  3032.24  2502.87  2233.3667  1326.5200  1.5334  100.0  100.3967  0.1235   

      8       9    ...     580       581     582     583     584      585  \
0  1.5005  0.0162  ...     NaN       NaN  0.5005  0.0118  0.0035   2.3630   
1  1.4966 -0.0005  ...  0.0060  208.2045  0.5019  0.0223  0.0055   4.4447   
2  1.4436  0.0041  ...  0.0148   82.8602  0.4958  0.0157  0.0039   3.1745   
3  1.4882 -0.0124  ...  0.0044   73.8432  0.4990  0.0103  0.0025   2.0544   
4  1.5031 -0.0031  ...     NaN       NaN  0.4800  0.4766  0.1045  99.3032   

      586     587     588       589  
0     Na

In [60]:
#master_df 형태
#|sample_id|timestamp|sensors(1 - 590)|label|


df_data = df_data2
df_label = df_label2

assert len(df_data) == len(df_label)
assert df_data.index.equals(df_label.index)

# feature_df schema 정리 -> ex) sensor_0 , sensor_1 ....
new_data_cols = [f"sensor_{i:03d}" for i in range(1, 591)]
df_data.columns = new_data_cols

# label_df schema 정리
new_label_cols = ["label", "timestamp"]
df_label.columns = new_label_cols
df_label["label"] = df_label["label"].astype("category") # to reduce memory by changing dtype as category
df_label["timestamp"] = pd.to_datetime(df_label["timestamp"]) # to sort by datetime in future


#check revised data and labels
# print(df_data[:5])
# print(df_label[:5])

# → dataframe merge
# → master_df 생성 merge(공통 키 있을떄 유리) vs concat(이어 붙힐떄)
master_df = pd.concat([df_data, df_label], axis = 1)


# Column 재정렬
cols = master_df.columns.to_list()
cols.remove("timestamp")
cols.insert(0, "timestamp")
master_df = master_df[cols]
# print(master_df[:5])

#sample id 생성후 삽입
master_df.insert(0,
                 "sample_id",
                 ["sample_" + str(i + 1).zfill(4) for i in range(len(master_df))])
# print(master_df[1500:])

#data/processed/ 데이터저장
data_processed_parquet_path = "drive/MyDrive/Semiconductor-AnomalyDetection/data/processed/master_df.parquet"
data_processed_csv_path = "drive/MyDrive/Semiconductor-AnomalyDetection/data/processed/master_df.csv"
master_df.to_parquet(data_processed_parquet_path, index=False)
master_df.to_csv(data_processed_csv_path, index=False)

temp = pd.read_parquet(data_processed_parquet_path)
print(temp.head())



     sample_id           timestamp  sensor_001  sensor_002  sensor_003  \
0  sample_0001 2008-07-19 11:55:00     3030.93     2564.00   2187.7333   
1  sample_0002 2008-07-19 12:32:00     3095.78     2465.14   2230.4222   
2  sample_0003 2008-07-19 13:17:00     2932.61     2559.94   2186.4111   
3  sample_0004 2008-07-19 14:43:00     2988.72     2479.90   2199.0333   
4  sample_0005 2008-07-19 15:22:00     3032.24     2502.87   2233.3667   

   sensor_004  sensor_005  sensor_006  sensor_007  sensor_008  ...  \
0   1411.1265      1.3602       100.0     97.6133      0.1242  ...   
1   1463.6606      0.8294       100.0    102.3433      0.1247  ...   
2   1698.0172      1.5102       100.0     95.4878      0.1241  ...   
3    909.7926      1.3204       100.0    104.2367      0.1217  ...   
4   1326.5200      1.5334       100.0    100.3967      0.1235  ...   

   sensor_582  sensor_583  sensor_584  sensor_585  sensor_586  sensor_587  \
0         NaN      0.5005      0.0118      0.0035      2.

In [ ]:
#1. 데이터 구조 확인

# 1. 데이터 shape 확인

# 2. 컬럼 개수 확인
# 3. 데이터 타입 확인
# 4. 첫 5개 row 출력
# 5. describe() 실행

In [ ]:
#2. Label 분석
# 1. label 값 종류 확인
# 2. 정상/이상 데이터 개수 계산
# 3. 정상/이상 데이터 비율 계산
# 4. class imbalance 여부 판단

In [ ]:
#3. 결측치 분석

# 1. 센서별 결측치 개수 계산
# 2. 센서별 결측치 비율 계산
# 3. 결측치가 하나라도 존재하는 센서 개수 확인
# 4. 결측치 비율 상위 10개 센서 출력
# 5. 결측치 비율 히스토그램 확

In [ ]:
# 4. 센서 값 분포 분석

# 1. 센서 값 분포 히스토그램 확인 (랜덤 센서 5~10개 선택)
# 2. 평균 / 중앙값 / 표준편차 비교
# 3. 극단값(outlier) 존재 여부 확인
# 4. 분포가 심하게 치우친 센서 확인

In [ ]:
#5. 정상 vs 이상 데이터 비교
# 1. 정상 데이터 subset 생성
# 2. 이상 데이터 subset 생성
# 3. 특정 센서에서 정상 vs 이상 분포 비교
# 4. 평균 차이가 큰 센서 찾기
# 5. 분산 차이가 큰 센서 찾기

In [ ]:
# 6. 센서 상관관계 비교

# 1. feature correlation matrix 계산
# 2. correlation이 높은 feature pair 찾기
# 3. correlation heatmap 시각화
# 4. 중복 정보 가진 센서 후보 찾기

In [ ]:
# 7. 저분산 Feature 분석

# 1. 센서별 분산 계산
# 2. 분산이 매우 작은 센서 찾기
# 3. 정보량이 낮은 feature 후보 정리

In [ ]:
# 8. Feature Scaling 분석

# 1. 센서별 값 범위(min/max) 비교
# 2. scale 차이가 큰 센서 확인
# 3. scaling 필요 여부 판단

In [ ]:
# 9. 이상치 후보 탐색

# 1. Z-score 기반 이상치 탐색
# 2. IQR 기반 이상치 탐색
# 3. 이상치 비율이 높은 센서 확인

In [ ]:
# 10. 전처리 전략 초안 작성

# 1. 제거할 센서 후보 리스트 작성
# 2. 결측치 처리 방법 후보 정리
# 3. scaling 방법 후보 정리
# 4. feature selection 필요 여부 판단

In [ ]:
#11. Anomaly 탐지 난이도 분석

# 1. 정상 데이터만 PCA 수행
# 2. 정상 데이터 분포 시각화
# 3. 이상 데이터 PCA projection 비교
# 4. anomaly가 separable한지 판단

improvement_log.md[EDA]

- 결측치 시각화 코드 가독성 낮음
- 변수명 모호함
- plot 함수 재사용 가능

[전처리]

- scaling 위치 고민 필요